In [ ]:
!pip install sentence-transformers
from google.colab import files
uploaded_files = files.upload() # upload the local files from the computer

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0,'/content') # looking in /content for any local files we uploaded

In [ ]:
df = pd.read_csv('data_preprocessed_csv.csv')
df['cleaned_abstract'] = df['cleaned_abstract'].fillna('') # handling missing values
df['abstract'] = df['abstract'].fillna('')
print("Dataset dimensions :",df.shape)
print("Columns in dataset :",df.columns.values)
print("\nFirst 2 rows of the abstract column :")
print(df['abstract'].head())

In [ ]:
period_order = ['2007-2011','2012-2016','2017-2021'] #standardising the range to compare with others later

# Using MiniLM Model

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model_used = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# Extracts abstracts and encode them into numerical embeddings using the model
abstracts = df['abstract'].tolist()
embeddings = model_used.encode(abstracts,show_progress_bar=True)
print(f"Embeddings shape : {embeddings.shape}")

In [ ]:
np.save('embeddings_nithya.npy',embeddings) # save the embeddings to the computer
files.download('embeddings_nithya.npy')

In [ ]:
def manual_cossim(ppr1,ppr2):
    return np.dot(ppr1,ppr2)/(np.linalg.norm(ppr1)*np.linalg.norm(ppr2))  # Implementing cosine similarity manually like aita lab

# Picking 2 random papers and checking their similarity score
ppr1 = embeddings[0]
ppr2 = embeddings[1]
similarity_score = manual_cossim(ppr1,ppr2)
print("\nPaper 1 is about :",df['abstract'].iloc[0])
print("\nPaper 2 is about :",df['abstract'].iloc[1])
print(f"Their similarity score is : {similarity_score : }") # Range betwen 0 to 1, higher means more similar

In [ ]:
def table_of_similar_papers(query,top_k=10):
    query_embdng = model_used.encode([query],convert_to_numpy=True)[0]
    scores = []
    for e in embeddings:
        score = manual_cossim(query_embdng,e)
        scores.append(score)
    scores = np.array(scores)
    top_results = scores.argsort()[::-1][:top_k]
    results = df.iloc[top_results][['title','year','abstract']].copy()
    results['similarity score'] = scores[top_results].round(4)
    results['abstract'] = results['abstract'].str[:300]
    return results.reset_index(drop=True)

In [ ]:
table_of_similar_papers("deep learning")

In [ ]:
table_of_similar_papers("Natural language processing")

In [ ]:
from sklearn.cluster import KMeans

number_of_clusters = 10 # using 10 as k for now, will be trying different k vals laterS
kmeans_clustering = KMeans(n_clusters=number_of_clusters,random_state=33)
df['kmeans_clusters'] = kmeans_clustering.fit_predict(embeddings)
print("Number of papers in each cluster : \n")
print(df['kmeans_clusters'].value_counts().sort_index())

In [ ]:
print("Few paper titles from each cluster :\n")
for c in range(number_of_clusters):
    paper_titles = df[df['kmeans_clusters'] == c]['title'].head(10)
    print(f"Few Papers in the Cluster {c} :")
    print(paper_titles.to_string(index=False))
    print()

In [ ]:
import matplotlib.pyplot as plt

X_matrix = embeddings
mu = X_matrix.mean(axis=0)
Z = X_matrix - mu

cov_matrix = np.cov(Z,rowvar=False)

eigvals, eigvecs = np.linalg.eigh(cov_matrix)

idx = np.argsort(eigvals)[::-1]
eigvecs = eigvecs[:,idx]
pca_projection = Z @ eigvecs[:,:2]

df['pca_1'] = pca_projection[:,0]
df['pca_2'] = pca_projection[:,1]

fig, ax = plt.subplots(figsize=(15,7))
scatter = ax.scatter(df['pca_1'],df['pca_2'],c=df['kmeans_clusters'],s=2)
plt.colorbar(scatter,ax=ax,label='Clusters')
ax.set_title('PCA Clusters - Sbert Embeddings',fontsize=12)
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
plt.savefig('pca_clusters_nithya_sbert.png',dpi=150)
files.download('pca_clusters_nithya_sbert.png')
plt.show()

In [ ]:
print(df.head())

In [ ]:
print(df['period'].value_counts().sort_index())

In [ ]:
print("Oldest Year :",df['year'].min())
print("Latest Year :",df['year'].max())
print("Total num of papers :",len(df))

# Experimenting with another model - Specter

In [ ]:
model_specter = SentenceTransformer('sentence-transformers/allenai-specter')  #using specter model to compare with MiniLM

In [ ]:
similarity_score_comparision = model_specter.encode([df['abstract'].iloc[0], df['abstract'].iloc[1]],convert_to_numpy=True)
specter_similarity_score = manual_cossim(similarity_score_comparision[0],similarity_score_comparision[1])
print(f"MiniLM similarity score : {similarity_score :}")
print(f"Specter similarity score : {specter_similarity_score :}")

In [ ]:
specter_model_embeddings = model_specter.encode(df['abstract'].tolist(),show_progress_bar=True)
print(specter_model_embeddings.shape)

In [ ]:
np.save('specter_model_embeddings_nithya.npy',specter_model_embeddings)
files.download('specter_model_embeddings_nithya.npy')

In [ ]:
from sklearn.metrics import silhouette_score

k_means_test = KMeans(n_clusters=10,random_state=33)

minilm_label = k_means_test.fit_predict(embeddings)
minilm_similarity_score = silhouette_score(embeddings,minilm_label,metric='cosine',sample_size=3000)

specter_label = k_means_test.fit_predict(specter_model_embeddings)
specter_similarity_score = silhouette_score(specter_model_embeddings,specter_label,metric='cosine',sample_size=3000)

print(f"MiniLM silhouette score : {minilm_similarity_score:}")
print(f"Specter silhouette score : {specter_similarity_score:}")

In [ ]:
# trying different k values

k_vals = []
for v in range(5,20):
    k_vals.append(v)

minilm_similarity_score_k = []
specter_similarity_score_k = []

for k in k_vals:
    k_means = KMeans(n_clusters=k,random_state=33)
    minilm_labels_k = k_means.fit_predict(embeddings)
    minilm_similarity_score_k.append(silhouette_score(embeddings,minilm_labels_k,metric='cosine',sample_size=3000))
    specter_labels_k = k_means.fit_predict(specter_model_embeddings)
    specter_similarity_score_k.append(silhouette_score(specter_model_embeddings,specter_labels_k,metric='cosine',sample_size=3000))
    print(f"k = {k}  MiniLM = {minilm_similarity_score_k[-1]:.4f}  Specter = {specter_similarity_score_k[-1]:.4f}")

In [ ]:
# plotting the silhouette scores

plt.figure(figsize=(10,6))
plt.plot(k_vals,minilm_similarity_score_k,marker='o',label='MiniLM')
plt.plot(k_vals,specter_similarity_score_k,marker='o',label='Specter')
plt.xlabel('K-vals')
plt.ylabel('Silhouette score')  # higher the better
plt.title('Silhouette Score Comparison - MiniLM VS Specter')
plt.legend()
plt.grid(True)
plt.savefig('silhouette_comparison_minilm_vs_specter.png',dpi=150)
files.download('silhouette_comparison_minilm_vs_specter.png')
plt.show()

# Using one more model MPNet for comparision


In [ ]:
model_mpnet = SentenceTransformer('all-mpnet-base-v2')  # experimenting with another model MPNet

In [ ]:
mpnet_model_embeddings = model_mpnet.encode(df['abstract'].tolist(),show_progress_bar=True)
print(mpnet_model_embeddings.shape)
np.save('mpnet_model_embeddings_nithya.npy',mpnet_model_embeddings)
files.download('mpnet_model_embeddings_nithya.npy')

In [ ]:
embeddings_dict = {'MiniLM':embeddings,'MPNet':mpnet_model_embeddings,'Specter':specter_model_embeddings} # storing all 3 model's embeddings as a dict for easier access
for model_name,embdngs in embeddings_dict.items():
    print(model_name,embdngs.shape) # to verify if all 3 gives same num of rows like it should

In [ ]:
print("Silhouette Scores for all 3 models :")  # comparing all 3 models
for model_name,embdngs in embeddings_dict.items():
    k_means = KMeans(n_clusters=10,random_state=33)
    labels = k_means.fit_predict(embdngs)
    silh_score = silhouette_score(embdngs,labels,metric='cosine',sample_size=3000)
    print(f"{model_name} : {silh_score:.4f}")

# Can the models tell the time periods apart?

In [ ]:
from sklearn.preprocessing import LabelEncoder

period_nums = LabelEncoder().fit_transform(df['period'])
for model_name,embdngs in embeddings_dict.items():
    silh_score = silhouette_score(embdngs,period_nums,metric='cosine',sample_size=9000)
    print(f"{model_name} : {silh_score:.4f}")

In [ ]:
# to save silhouette scores

silhouette_results = {}
for model_name,embdngs in embeddings_dict.items():
    silh_score = silhouette_score(embdngs,period_nums,metric='cosine',sample_size=9000)
    silhouette_results[model_name] = round(float(silh_score),6)
pd.DataFrame([silhouette_results]).to_csv('sbert_silhouette_scores_nithya.csv',index=False)
files.download('sbert_silhouette_scores_nithya.csv')

print(silhouette_results)

# PCA

In [ ]:
X_spec = specter_model_embeddings
mu_spec = X_spec.mean(axis=0)
Z_spec = X_spec - mu_spec

R_spec = np.cov(Z_spec,rowvar=False)
evals_spec, evecs_spec = np.linalg.eigh(R_spec)

idx_spec = np.argsort(evals_spec)[::-1]
evecs_spec = evecs_spec[:,idx_spec]

pca_coords = Z_spec @ evecs_spec[:,:2]

df['pca_x'] = pca_coords[:,0]
df['pca_y'] = pca_coords[:,1]

total_var = np.sum(evals_spec)
print(f"PC 1 : {(evals_spec[idx_spec[0]]/total_var)*100:.2f}%")
print(f"PC 2 : {(evals_spec[idx_spec[1]]/total_var)*100:.2f}%")

In [ ]:
fig,ax = plt.subplots(figsize=(10,7))

for p in period_order:
    mask = df['period'] == p
    ax.scatter(df.loc[mask,'pca_x'],df.loc[mask,'pca_y'],label=p,s=2)

ax.set_title('PCA of Specter embeddings by time period')
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.legend(markerscale=6)
plt.savefig('pca_by_period_nithya.png',dpi=150)
files.download('pca_by_period_nithya.png')
plt.show()

In [ ]:
# Export PCA coordinates for integration notebook

pca_export = df[['period','pca_x','pca_y']].copy()
pca_export.to_csv('sbert_pca_coords_nithya.csv',index=False)
files.download('sbert_pca_coords_nithya.csv')
print(pca_export.shape)

# Visualising the Trajectory of the field

In [ ]:
centroid_x_coord = []
centroid_y_coord = []

for p in period_order:
    mask = df['period'] == p
    mean_x = df.loc[mask,'pca_x'].mean()
    mean_y = df.loc[mask,'pca_y'].mean()
    centroid_x_coord.append(mean_x)
    centroid_y_coord.append(mean_y)
    print(f"Centroid for {p} : ({mean_x:.4f},{mean_y:.4f})")

In [ ]:
plt.figure(figsize=(12,8))
plt.scatter(df['pca_x'],df['pca_y'],s=1,color='grey')
plt.plot(centroid_x_coord,centroid_y_coord,'k--',linewidth=1) # dashed line connecting centroids to show change

for i, p in enumerate(period_order):
    plt.scatter(centroid_x_coord[i],centroid_y_coord[i],s=150)
    plt.text(centroid_x_coord[i]+0.1,centroid_y_coord[i]+0.1,p,fontsize=10,color='black')

plt.title('Trajectory of Research Topics')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.grid(True)
plt.savefig('trajectory_topics_nithya.png',dpi=150)
files.download('trajectory_topics_nithya.png')
plt.show()

# By how much the field changed between each period


In [ ]:
space_row = []

for model_name,embdngs in embeddings_dict.items():
    centroids = {}
    for p in period_order:
        mask = df['period'] == p
        centroids[p] = embdngs[mask.values].mean(axis=0)
    for i in range(len(period_order)-1):
        p1 = period_order[i]
        p2 = period_order[i+1]
        cos_sim = manual_cossim(centroids[p1],centroids[p2])
        space_row.append({'model':model_name,'period from':p1,'period to':p2,'distance':round(1-cos_sim,4)})

space_table = pd.DataFrame(space_row)
print(space_table)

In [ ]:
# to save model's centroid distances

# MiniLM
dist_minilm = space_table[space_table['model']=='MiniLM']['distance'].tolist()
np.save('sbert_minilm_centroid_distances_nithya.npy',dist_minilm)
files.download('sbert_minilm_centroid_distances_nithya.npy')
print(f"MiniLM distances saved : {dist_minilm}")

# MPNet
dist_mpnet = space_table[space_table['model']=='MPNet']['distance'].tolist()
np.save('sbert_mpnet_centroid_distances_nithya.npy',dist_mpnet)
files.download('sbert_mpnet_centroid_distances_nithya.npy')
print(f"MPNet distances saved : {dist_mpnet}")

# Specter
dist_specter = space_table[space_table['model']=='Specter']['distance'].tolist()
np.save('sbert_specter_centroid_distances_nithya.npy',dist_specter)
files.download('sbert_specter_centroid_distances_nithya.npy')
print(f"Specter distances saved : {dist_specter}")

In [ ]:
transitions = [f"{period_order[i]} to {period_order[i+1]}" for i in range(len(period_order) - 1)]

fig, ax = plt.subplots(figsize=(12,5))
bar_width = 0.25
x = np.arange(len(transitions))

for i,model_name in enumerate(['MiniLM','MPNet','Specter']):
    rows = space_table[space_table['model'] == model_name]
    ax.bar(x+i*bar_width,rows['distance'].tolist(),bar_width,label=model_name)

ax.set_xticks(x+bar_width)
ax.set_xticklabels(transitions)
ax.set_ylabel('Cosine distance')
ax.set_title('Field changed between periods')
ax.legend()
plt.savefig('field_changed_between_periods_nithya.png',dpi=150)
files.download('field_changed_between_periods_nithya.png')
plt.show()

# Analysing how communication and meaning has changed

In [ ]:
def find_pprs_for_term(description,period,top_k=8):
    term_emb = model_specter.encode([description],convert_to_numpy=True)[0]
    mask = df['period'] == period
    period_df = df[mask].reset_index(drop=True)
    period_embdngs = specter_model_embeddings[mask.values]
    scores = []
    for pe in period_embdngs:
        cos_sim = manual_cossim(term_emb,pe)
        scores.append(cos_sim)
    scores = np.array(scores)
    top_idx = scores.argsort()[-top_k:][::-1]
    results = period_df.iloc[top_idx][['year','title']].copy()
    results['similarity_score'] = scores[top_idx].round(3)
    return results.reset_index(drop=True)

In [ ]:
print("'language models' across periods:\n")

for p in period_order:
    print(f"{[p]} :")
    r = find_pprs_for_term('a model that predicts and generates natural language text',p)
    print(r[['year','title']].to_string(index=False))
    print()

In [ ]:
# trying another one

print("'neural network' across periods:\n")

for p in period_order:
    print(f"{[p]} :")
    r = find_pprs_for_term('deep neural network architecture',p)
    print(r[['year','title']].to_string(index=False))
    print()

# Analysing how focus has changed


In [ ]:
# measuring average distance within each period

diversity_rows_period = []

for model_name,embdngs in embeddings_dict.items():
    for p in period_order:
        mask = df['period'] == p
        period_embdngs = embdngs[mask.values]
        centroid = period_embdngs.mean(axis=0)
        if len(period_embdngs)>1000:
            choice = np.random.choice(len(period_embdngs),1000,replace=False)
            sample_embdngs = period_embdngs[choice]
        else:
            sample_embdngs = period_embdngs
        dist = []
        for se in sample_embdngs:
            cos_sim = manual_cossim(se,centroid)
            dist.append(1 - cos_sim)
        avg_diversity = np.mean(dist)
        diversity_rows_period.append({'model': model_name,'period': p,'diversity_score': round(avg_diversity,4)})

diversity_table = pd.DataFrame(diversity_rows_period)
print(diversity_table)

In [ ]:
fig,ax = plt.subplots(figsize=(10,4))

for model_name in ['MiniLM','MPNet','Specter']:
    rows = diversity_table[diversity_table['model'] == model_name]
    rows = rows.set_index('period').reindex(period_order)
    ax.plot(period_order,rows['diversity_score'].values,marker='o',label=model_name)

ax.set_xlabel('period')
ax.set_ylabel('Mean Cosine Distance')
ax.set_title('Research Diversity over Time')
ax.legend()
ax.grid(True)
plt.savefig('diversity_over_time_nithya.png',dpi=150)
files.download('diversity_over_time_nithya.png')
plt.show()

# Analysing how framing has changed

In [ ]:
eg_sentence = {'state-of-the-art': 'Our approach achieves state-of-the-art results on standard benchmarks',
             'theoretical': 'We provide a formal analysis of properties and establish theoretical results',
             'real-world appln': 'We demonstrate the practical application of our framework in a real-world scenario',
             'new dataset': 'We present a novel, large-scale annotated dataset',
             'scaling up': 'We scale our model to billions of parameters',
             'interpretability': 'We analyse what the model has learned internally'}

names = list(eg_sentence.keys())
framing_embdngs = model_specter.encode(list(eg_sentence.values()),convert_to_numpy=True)
print(framing_embdngs.shape)

In [ ]:
results = {}

for p in period_order:
    mask = df['period'] == p
    period_embdngs = specter_model_embeddings[mask.values]
    if len(period_embdngs)>800:
        choice = np.random.choice(len(period_embdngs),800,replace=False)
        p_embdngs = period_embdngs[choice]
    else:
        p_embdngs = period_embdngs
    period_scores = []
    for framing_emb in framing_embdngs:
        sims = []
        for p_emb in p_embdngs:
            sims.append(manual_cossim(framing_emb,p_emb))
        period_scores.append(np.mean(sims))
    results[p] = np.array(period_scores)

result_table = pd.DataFrame(results,index=names)
print(result_table.round(4))

In [ ]:
fig,ax = plt.subplots(figsize=(11,11))

image = ax.imshow(result_table.values,cmap='Purples')

ax.set_xticks(range(len(period_order)))
ax.set_xticklabels(period_order)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)

ax.set_title('Framing style change over Time')
plt.savefig('framing_style_heatmap_nithya.png',dpi=150)
files.download('framing_style_heatmap_nithya.png')
plt.show()

In [ ]:
# Exporting framing heatmap data for final integration
result_table.to_csv('sbert_framing_heatmap_nithya.csv')
files.download('sbert_framing_heatmap_nithya.csv')
